# Mens rea summary statistics

Standalone program. Reads every offense CSV in `DATA_DIR` (columns `law_number`, `conduct_merged`, `result_merged`, `circumstance_merged`) and reports aggregate statistics pooled across all offenses (no offense-by-offense breakdown). A coded value of `n/a` is treated as **Silent**.

It produces:
1. detailed tables in the notebook (overall distribution, by element, statute-level coverage, spread across jurisdictions, variation within a statute) and a written summary generated from them; and
2. a one-page **summary table** (Panel A by element, Panel B by statute), saved as PNG and SVG, with the underlying numbers saved as CSV.

**Scales automatically.** Counts of offenses, jurisdictions and cells are all taken from the data, so adding or removing CSVs needs no code changes.

**To run:** put the CSVs in the same folder as this notebook (or set `DATA_DIR`), then *Run All*. Output goes to a `figures` folder next to this notebook.

Requires `pandas`, `numpy`, `matplotlib` (and `jinja2`, which comes with Jupyter, for the styled tables).

In [ ]:
import re
import textwrap
from itertools import groupby
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
from IPython.display import display, Markdown

%matplotlib inline
pd.set_option("display.width", 160)

# ---- input / output -----------------------------------------------------------------
DATA_DIR = Path(".")                               # folder containing the offense CSVs
FILE_GLOB = "*_rounded_down.csv"                   # which files count as offense files
NAME_PATTERN = r"-sol_(.+)_rounded_down\.csv$"     # offense name = the part captured in ( )
OUT_DIR = Path("figures")                          # output folder, created next to this notebook
OUT_DIR.mkdir(exist_ok=True)

# ---- optional sanity checks (None = accept whatever is in the folder) ----------------
EXPECTED_OFFENSES = None          # e.g. 50 -> stop with an error if a different number is read
EXPECTED_JURISDICTIONS = None     # e.g. 51

# ---- coding scheme --------------------------------------------------------------------
ELEMENTS = {                      # CSV column -> label, in display order
    "conduct_merged": "Conduct",
    "result_merged": "Result",
    "circumstance_merged": "Circumstance",
}
EL_LABELS = list(ELEMENTS.values())
LEVELS = ["Purposive", "Knowing", "Reckless", "Negligent", "Strict Liability", "Silent"]
STATED_LEVELS = LEVELS[:-1]
CULPABILITY_SCORE = {"Purposive": 4, "Knowing": 3, "Reckless": 2,
                     "Negligent": 1, "Strict Liability": 0}

# Same palette as the bar graphs (used for the colour swatches in the summary table)
COLORS = {
    "Purposive":        "#7A0030",
    "Knowing":          "#B8004A",
    "Reckless":         "#E6005D",
    "Negligent":        "#F27DA8",
    "Strict Liability": "#F9C2D5",
    "Silent":           "#D6D6D6",
}

# ---- summary table appearance -----------------------------------------------------------
TABLE_WIDTH = 8.5      # inches (letter width); the height trims itself to the content
FONT = 7               # base font size (pt)


In [ ]:
SMALL_WORDS = {"a", "an", "the", "to", "with", "of", "and", "or", "in", "on", "for"}

def display_name(stem: str) -> str:
    """'Driving_with_a_Suspended_License' -> 'Driving with a Suspended License',
    'Kidnapping_1st_degree' -> 'Kidnapping 1st Degree', 'DUI' stays 'DUI'."""
    words = stem.replace("_", " ").split()
    return " ".join(
        w.lower() if (i > 0 and w.lower() in SMALL_WORDS) else w[0].upper() + w[1:]
        for i, w in enumerate(words)
    )

# ---- read every matching file ----------------------------------------------------------
frames, skipped = [], []
for path in sorted(DATA_DIR.glob(FILE_GLOB)):
    m = re.search(NAME_PATTERN, path.name)
    if not m:
        skipped.append(path.name)
        continue
    # keep_default_na=False keeps the literal string "n/a" instead of turning it into NaN
    df = pd.read_csv(path, keep_default_na=False)
    missing_cols = {"law_number", *ELEMENTS} - set(df.columns)
    if missing_cols:
        raise ValueError(f"{path.name} is missing columns: {sorted(missing_cols)}")
    df["offense"] = display_name(m.group(1))
    df["source_file"] = path.name
    frames.append(df)

if skipped:
    print(f"WARNING: skipped {len(skipped)} file(s) whose names don't match NAME_PATTERN:\n  "
          + "\n  ".join(skipped))
if not frames:
    raise FileNotFoundError(f"No offense files matching {FILE_GLOB!r} in {DATA_DIR.resolve()}")

raw = pd.concat(frames, ignore_index=True)

# ---- consistency checks (all derived from the data, nothing hard-coded) ---------------
clash = raw.groupby("offense")["source_file"].unique().loc[lambda s: s.map(len) > 1]
if not clash.empty:
    raise ValueError("Different files produce the same offense name:\n"
                     + "\n".join(f"  {o}: {list(f)}" for o, f in clash.items()))

dupe_rows = raw.groupby("offense")["law_number"].agg(lambda s: int(s.duplicated().sum())).loc[lambda s: s > 0]
if not dupe_rows.empty:
    raise ValueError(f"Repeated law_number rows within an offense:\n{dupe_rows.to_string()}")

ALL_JURIS = sorted(raw["law_number"].unique())
N_JURIS = len(ALL_JURIS)
coverage = raw.groupby("offense")["law_number"].nunique()
incomplete = coverage[coverage != N_JURIS]
if not incomplete.empty:
    raise ValueError(f"These offenses don't cover all {N_JURIS} jurisdictions "
                     f"(rows found shown):\n{incomplete.to_string()}")

N_OFFENSES = raw["offense"].nunique()
if EXPECTED_OFFENSES is not None and N_OFFENSES != EXPECTED_OFFENSES:
    raise ValueError(f"Expected {EXPECTED_OFFENSES} offenses, found {N_OFFENSES}")
if EXPECTED_JURISDICTIONS is not None and N_JURIS != EXPECTED_JURISDICTIONS:
    raise ValueError(f"Expected {EXPECTED_JURISDICTIONS} jurisdictions, found {N_JURIS}")

long = (
    raw.melt(id_vars=["offense", "law_number"], value_vars=list(ELEMENTS),
             var_name="element", value_name="mens_rea")
       .assign(element=lambda d: d["element"].map(ELEMENTS),
               mens_rea=lambda d: d["mens_rea"].astype(str).str.strip().replace({"n/a": "Silent"}))
)
long["stated"] = long["mens_rea"] != "Silent"

unknown = set(long["mens_rea"]) - set(LEVELS)
if unknown:
    raise ValueError(f"Unexpected mens rea labels: {sorted(unknown)}")

print(f"OK: {N_OFFENSES} offenses x {N_JURIS} jurisdictions x {len(ELEMENTS)} elements "
      f"= {len(long):,} coded element-cells")

all_silent = long.groupby("offense")["stated"].sum().loc[lambda s: s == 0].index.tolist()
if all_silent:
    print(f"WARNING: no stated mens rea anywhere in: {', '.join(all_silent)} "
          "(check whether this is real silence or an extraction problem)")


## Detailed tables

Units: an **element-cell** is one offense × one jurisdiction × one element; a **statute** is one offense in one jurisdiction.

In [ ]:
def show(df, caption):
    display(Markdown(f"**{caption}**"))
    display(df)

stated = long[long["stated"]].copy()
stated["score"] = stated["mens_rea"].map(CULPABILITY_SCORE)
N_EL = len(ELEMENTS)
N_CELLS, N_STATED = len(long), len(stated)
N_STATUTES = N_OFFENSES * N_JURIS
display(Markdown(f"{N_OFFENSES} offenses \u00d7 {N_JURIS} jurisdictions \u00d7 {N_EL} elements = "
                 f"**{N_CELLS:,} element-cells** in **{N_STATUTES:,} statutes**."))

# 1. Overall distribution
overall = long["mens_rea"].value_counts().reindex(LEVELS, fill_value=0).rename("cells").to_frame()
overall["% of all cells"] = overall["cells"] / N_CELLS * 100
overall["% of stated cells"] = (overall["cells"] / N_STATED * 100).where(overall.index != "Silent")
overall.loc["Total"] = [N_CELLS, 100.0, np.nan]
overall["cells"] = overall["cells"].astype(int)
show(overall.round(1).fillna(""), f"Table 1. Mens rea across all {N_CELLS:,} element-cells")

# 2. By element
by_el_all = (pd.crosstab(long["element"], long["mens_rea"], normalize="index") * 100) \
    .reindex(index=EL_LABELS, columns=LEVELS, fill_value=0)
show(by_el_all.round(1),
     f"Table 2a. Distribution by element, % of all cells ({N_STATUTES:,} per element)")

by_el_stated = (pd.crosstab(stated["element"], stated["mens_rea"], normalize="index") * 100) \
    .reindex(index=EL_LABELS, columns=STATED_LEVELS, fill_value=0)
by_el_stated.insert(0, "stated cells", stated.groupby("element").size().reindex(EL_LABELS, fill_value=0))
by_el_stated["mean culpability (SL=0 \u2026 P=4)"] = stated.groupby("element")["score"].mean().reindex(EL_LABELS)
show(by_el_stated.round(2), "Table 2b. Distribution by element, % of stated cells only")


In [ ]:
# 3. Statute-level coverage: how many elements carry a stated mens rea?
statute_stated = long.groupby(["offense", "law_number"])["stated"].sum()
coverage = statute_stated.value_counts().reindex(range(N_EL + 1), fill_value=0).rename("statutes").to_frame()
coverage.index.name = f"elements stated (of {N_EL})"
coverage["% of statutes"] = coverage["statutes"] / N_STATUTES * 100
show(coverage.round(1), f"Table 3. Statute-level coverage ({N_STATUTES:,} statutes)")

# 4. Spread across jurisdictions: stated cells per jurisdiction
cells_per_juris = N_OFFENSES * N_EL
by_juris = long.groupby("law_number")["stated"].sum()
juris_desc = (by_juris.describe().rename({"count": "jurisdictions"}).to_frame(f"stated cells (of {cells_per_juris})").T)
show(juris_desc.round(1), "Table 4. Stated cells per jurisdiction")

# 5. Element-by-element variation inside a single statute
per_statute = stated.groupby(["offense", "law_number"])["mens_rea"].agg(n="size", distinct="nunique")
multi = per_statute[per_statute["n"] >= 2]
variation = pd.DataFrame({
    "statutes": [int((multi["distinct"] == 1).sum()), int((multi["distinct"] > 1).sum())],
}, index=["same level on every stated element", "different levels across elements"])
variation["%"] = variation["statutes"] / len(multi) * 100 if len(multi) else np.nan
show(variation.round(1), f"Table 5a. Statutes with 2+ stated elements (n = {len(multi):,})")

full = raw.assign(**{c: raw[c].astype(str).str.strip() for c in ELEMENTS})
full = full[(full[list(ELEMENTS)] != "n/a").all(axis=1)]
patterns = (full.groupby(list(ELEMENTS)).size().rename("statutes")
                .sort_values(ascending=False).head(8).reset_index()
                .rename(columns=ELEMENTS))
patterns["% of fully stated"] = patterns["statutes"] / len(full) * 100 if len(full) else np.nan
show(patterns.round(1),
     f"Table 5b. Most common patterns among statutes stating all {N_EL} elements (n = {len(full):,})")


In [ ]:
# Written summary, generated from the tables above (wording adapts to whatever the data show)
pct = lambda n, d: 100 * n / d if d else float("nan")
n_silent = N_CELLS - N_STATED
lvl = overall.loc[STATED_LEVELS, "% of stated cells"].astype(float).sort_values(ascending=False)
el_stated_pct = (long.groupby("element")["stated"].mean() * 100).reindex(EL_LABELS).sort_values(ascending=False)
el_mean = by_el_stated["mean culpability (SL=0 \u2026 P=4)"].sort_values(ascending=False)
el_mode = by_el_stated[STATED_LEVELS].idxmax(axis=1)
cov = coverage["% of statutes"]

excl_note = ""
if all_silent:
    n_with_data = N_OFFENSES - len(all_silent)
    verb = "has" if len(all_silent) == 1 else "have"
    excl_note = (f" {', '.join(all_silent)} {verb} no stated mens rea at all; excluding "
                 f"{'it' if len(all_silent) == 1 else 'them'}, the stated share is "
                 f"{pct(N_STATED, n_with_data * N_JURIS * N_EL):.1f}%.")

join = lambda items: ", ".join(items[:-1]) + (" and " if len(items) > 1 else "") + items[-1]

summary = f"""
**Coverage.** Of {N_CELLS:,} element-cells ({N_OFFENSES} offenses \u00d7 {N_JURIS} jurisdictions \u00d7 {N_EL} elements),
{N_STATED:,} ({pct(N_STATED, N_CELLS):.1f}%) state a mens rea and {n_silent:,} ({pct(n_silent, N_CELLS):.1f}%) are silent.{excl_note}

**Which levels.** Among stated cells: {join([f"{l} {v:.1f}%" for l, v in lvl.items()])}.

**By element.** Share of cells with a stated mens rea: {join([f"{e} {v:.1f}%" for e, v in el_stated_pct.items()])}.
Mean culpability when stated (0\u20134 scale): {join([f"{e} {v:.2f}" for e, v in el_mean.items()])}.
Most common stated level: {join([f"{l} for {e.lower()} ({by_el_stated.loc[e, l]:.1f}%)" for e, l in el_mode.items()])}.

**By statute.** Of {N_STATUTES:,} offense-jurisdiction statutes, {cov.loc[N_EL]:.1f}% state a mens rea for all {N_EL} elements,
{join([f"{cov.loc[k]:.1f}% for {k}" for k in range(N_EL - 1, 0, -1)] + [f"{cov.loc[0]:.1f}% for none"])}.
Among statutes with two or more stated elements, {variation.loc["different levels across elements", "%"]:.1f}% assign
different levels to different elements.

**By jurisdiction.** Stated cells per jurisdiction range from {by_juris.min()} to {by_juris.max()} of {cells_per_juris}
(median {by_juris.median():.0f}).
"""
display(Markdown(summary))


## Summary table

Panel **A** by element and Panel **B** at the statute level, pooled across all offenses. Displayed below, saved as CSV, and rendered as a PNG and SVG.

In [ ]:
def level_shares(labels):
    # % of stated cells at each level; NaN if nothing is stated
    if len(labels) == 0:
        return pd.Series(np.nan, index=STATED_LEVELS)
    return labels.value_counts(normalize=True).reindex(STATED_LEVELS, fill_value=0) * 100

def mean_culpability(labels):
    return labels.map(CULPABILITY_SCORE).mean() if len(labels) else np.nan

COV, LVL = "Coverage", "Level among stated cells (%)"

# ---- Panel A: by element ---------------------------------------------------------
rows = {}
for el in EL_LABELS + ["All elements"]:
    sub = long if el == "All elements" else long[long["element"] == el]
    st = sub.loc[sub["stated"], "mens_rea"]
    row = {(COV, "Cells (n)"): len(sub), (COV, "Stated (n)"): len(st),
           (COV, "Stated (%)"): 100 * len(st) / len(sub), (COV, "Silent (%)"): 100 * (1 - len(st) / len(sub))}
    row.update({(LVL, l): v for l, v in level_shares(st).items()})
    row[("", "Mean culpability (0\u20134)")] = mean_culpability(st)
    rows[el] = row
panel_a = pd.DataFrame(rows).T
panel_a.columns = pd.MultiIndex.from_tuples(panel_a.columns)

# ---- Panel B: statute level (offense x jurisdiction) -----------------------------
EST = "Elements with stated mens rea (% of statutes)"
row = {("", "Statutes (n)"): N_STATUTES}
row.update({(EST, "0 (fully silent)" if k == 0 else str(k)): v for k, v in
            (statute_stated.value_counts(normalize=True).reindex(range(N_EL + 1), fill_value=0) * 100).items()})
row[("", "Mixed levels (% of statutes with 2+ stated)")] = variation.loc["different levels across elements", "%"]
panel_b = pd.DataFrame({"All statutes": row}).T
panel_b.columns = pd.MultiIndex.from_tuples(panel_b.columns)

# ---- display + CSV -------------------------------------------------------------------
def fmt_map(df):
    return {c: ("{:,.0f}" if "(n)" in c[1] else "{:.2f}" if "culpability" in c[1] else "{:.1f}")
            for c in df.columns}

for name, df in [(f"A. By element (all {N_OFFENSES} offenses pooled)", panel_a),
                 ("B. Statute level (offense \u00d7 jurisdiction)", panel_b)]:
    display(Markdown(f"**Panel {name}**"))
    display(df.style.format(fmt_map(df), na_rep="\u2014"))

for tag, df in [("A_by_element", panel_a), ("B_statute_level", panel_b)]:
    flat = df.copy()
    flat.columns = [f"{g}: {c}" if g else c for g, c in df.columns]
    path = OUT_DIR / f"summary_table_{tag}.csv"
    flat.round(2).to_csv(path, index_label="")
    print(f"Saved {path.resolve()}")


In [ ]:
# Two-line versions of long headers for the rendered page
HEADER_WRAP = {
    "Circumstance": "Circum-\nstance", "All elements": "All\nelements", "Strict Liability": "Strict\nLiability",
    "Cells (n)": "Cells\n(n)", "Stated (n)": "Stated\n(n)", "Stated (%)": "Stated\n(%)", "Silent (%)": "Silent\n(%)",
    "Mean culpability (0\u20134)": "Mean\nculpability\n(0\u20134)",
    "Statutes (n)": "Statutes\n(n)", "0 (fully silent)": "0\n(fully\nsilent)",
    "Mixed levels (% of statutes with 2+ stated)": "Mixed levels\n(% of 2+\nstated)",
}

def fmt_cell(col, v):
    if pd.isna(v):
        return "\u2014"
    if "(n)" in col[1]:
        return f"{v:,.0f}"
    if "culpability" in col[1]:
        return f"{v:.2f}"
    return f"{v:.1f}"

def draw_panel(ax, y, label, df, x0, name_w, col_w, row_h=0.19, bold_rows=(), base=7):
    # Booktabs-style panel drawn in inch coordinates; returns the y position below it.
    cols = list(df.columns)
    x_end = x0 + name_w + len(cols) * col_w
    col_cx = lambda j: x0 + name_w + (j + 0.5) * col_w

    ax.text(x0, y, label, fontsize=base + 0.5, fontweight="bold", va="bottom")
    y += 0.07
    ax.plot([x0, x_end], [y, y], color="black", lw=0.9)

    # group headers with underline spans
    y += 0.15
    j = 0
    for grp, run in groupby(cols, key=lambda c: c[0]):
        n = len(list(run))
        if grp:
            left, right = x0 + name_w + j * col_w, x0 + name_w + (j + n) * col_w
            ax.text((left + right) / 2, y, grp, fontsize=base - 0.5, fontweight="bold", ha="center", va="center")
            ax.plot([left + 0.05, right - 0.05], [y + 0.09, y + 0.09], color="black", lw=0.5)
        j += n

    # sub-headers (up to 3 lines) with palette swatches under mens rea levels
    sub_h = 0.36
    y += 0.12
    for j, col in enumerate(cols):
        ax.text(col_cx(j), y + sub_h / 2, HEADER_WRAP.get(col[1], col[1]), fontsize=base - 0.5,
                ha="center", va="center", linespacing=1.05)
        if col[1] in COLORS:
            ax.add_patch(Rectangle((col_cx(j) - 0.3 * col_w, y + sub_h - 0.01), 0.6 * col_w, 0.045,
                                   color=COLORS[col[1]], lw=0))
    y += sub_h + 0.08
    ax.plot([x0, x_end], [y, y], color="black", lw=0.5)

    # body
    for name, row in df.iterrows():
        bold = name in bold_rows
        if bold:
            ax.plot([x0, x_end], [y, y], color="black", lw=0.5)
            ax.add_patch(Rectangle((x0, y), x_end - x0, row_h, color="#F2F2F2", lw=0, zorder=0))
        cy = y + row_h / 2
        weight = "bold" if bold else "normal"
        ax.text(x0 + 0.04, cy, name, fontsize=base, va="center", fontweight=weight)
        for j, col in enumerate(cols):
            ax.text(col_cx(j), cy, fmt_cell(col, row[col]), fontsize=base, ha="center", va="center",
                    fontweight=weight)
        y += row_h
    ax.plot([x0, x_end], [y, y], color="black", lw=0.9)
    return y

def render_summary_table(width=TABLE_WIDTH, base=FONT):
    W, H = width, 20                     # draw on a tall canvas, then trim to the content
    fig = plt.figure(figsize=(W, H))
    ax = fig.add_axes([0, 0, 1, 1])
    ax.set_xlim(0, W)
    ax.set_ylim(H, 0)                    # y runs downward, in inches
    ax.axis("off")

    x0, name_w = 0.5, 1.75
    col_w = (W - 2 * x0 - name_w) / max(len(panel_a.columns), len(panel_b.columns))

    ax.text(W / 2, 0.4,
            "Summary Statistics: Mens Rea by Element\n"
            f"pooled across {N_OFFENSES} offenses \u00d7 {N_JURIS} jurisdictions",
            fontsize=base * 1.6, fontweight="bold", ha="center", va="top", linespacing=1.15)

    y = 1.2
    y = draw_panel(ax, y, "A. By element", panel_a, x0, name_w, col_w,
                   bold_rows=["All elements"], base=base)
    y = draw_panel(ax, y + 0.32, "B. By statute (one offense in one jurisdiction)", panel_b, x0, name_w,
                   col_w, base=base)

    notes = (
        f"Notes: Each of {N_OFFENSES} offenses is coded for {N_JURIS} jurisdictions \u00d7 {N_EL} elements, giving "
        f"{N_CELLS:,} element-cells and {N_STATUTES:,} statutes. \u201cStated\u201d means any coded mens rea "
        "other than Silent (n/a). Level shares are percentages of stated cells. Mean culpability scores Strict "
        "Liability 0, Negligent 1, Reckless 2, Knowing 3, Purposive 4. A fully silent statute states no mens rea "
        "for any element. \u201cMixed levels\u201d = statutes with 2+ stated elements that assign different levels "
        "to different elements."
    )
    if all_silent:
        verb = "has" if len(all_silent) == 1 else "have"
        notes += f" {', '.join(all_silent)} {verb} no stated mens rea in any jurisdiction and {'is' if len(all_silent) == 1 else 'are'} included as Silent."
    wrapped = textwrap.fill(notes, int((W - 2 * x0) * 72 / (base * 0.9 * 0.54)))
    ax.text(x0, y + 0.15, wrapped, fontsize=base * 0.9, va="top", linespacing=1.3)

    # trim the canvas to the content (notes height ~ lines x line height)
    n_lines = wrapped.count("\n") + 1
    H_new = y + 0.15 + n_lines * base * 0.9 * 1.3 / 72 + 0.45
    fig.set_size_inches(W, H_new)
    ax.set_ylim(H_new, 0)
    return fig

fig = render_summary_table()
fig.savefig(OUT_DIR / "summary_table.png", dpi=300, facecolor="white")
fig.savefig(OUT_DIR / "summary_table.svg", facecolor="white")   # vector: stays sharp at any size
print(f"Saved {(OUT_DIR / 'summary_table').resolve()}.png and .svg")
plt.show()
